# Data Preparation

## Prepare torch for P100

In [1]:
!pip uninstall torch torchvision -y
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 104.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 88.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 241.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 114.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 236.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 214.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 162.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 129.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━

## Tokenizer

In [2]:
import pickle
import torch

class WubiCharTokenizer:
    """
    Tokenizes Chinese text into sequences of atomic Wubi symbols (letters + separator).
    Each character becomes a fixed number of tokens (its Wubi code letters + separator).
    No merging across characters occurs.
    """
    def __init__(self, wubi_dict_path):
        with open(wubi_dict_path, 'rb') as f:
            self.ch2wubi = pickle.load(f)

        # Build vocabulary: all possible letters + separator
        self.sep = '#'
        letters = set()
        for code in self.ch2wubi.values():
            letters.update(code)
        # Also include digits? Original mapping strips digits, so just letters.
        self.vocab = ['[PAD]', '[UNK]', self.sep] + sorted(letters)
        self.stoi = {s: i for i, s in enumerate(self.vocab)}
        self.itos = {i: s for s, i in self.stoi.items()}

    def encode(self, text):
        """
        Convert text to a list of token IDs and a list of character spans.
        Returns:
            ids: list of int (token IDs)
            char_spans: list of (start, end) token indices for each character
        """
        text = text.lower()
        ids = []
        char_spans = []
        for ch in text:
            # Get Wubi code (fallback to character itself)
            code = self.ch2wubi.get(ch, ch)
            # Convert each symbol in the code to an ID
            code_ids = [self.stoi.get(c, self.stoi['[UNK]']) for c in code]
            start = len(ids)
            ids.extend(code_ids)
            ids.append(self.stoi[self.sep])
            end = len(ids)
            char_spans.append((start, end))
        return ids, char_spans

    def decode(self, ids):
        """For debugging: convert IDs back to string."""
        return ''.join(self.itos[i] for i in ids if i != self.stoi[self.sep])

In [3]:
# Load the Wubi mapping (from the SubCharTokenization repo)
wubi_dict = ""
tokenizer = WubiCharTokenizer(wubi_dict)

text = "我是最高的学生"
ids, spans = tokenizer.encode(text)

print("Token IDs:", ids)
print("Character spans:", spans)
# Output: spans like [(0,5), (5,8), (8,15), ...] depending on code lengths

Token IDs: [29, 2, 22, 2, 22, 14, 2, 37, 25, 2, 30, 2, 21, 28, 2, 32, 19, 2]
Character spans: [(0, 2), (2, 4), (4, 7), (7, 10), (10, 12), (12, 15), (15, 18)]


In [4]:
# Inverse mapping from Wubi to Character
with open("", 'rb') as f:
    wubi2ch = pickle.load(f)

## Pinyin Mapper

In [5]:
INITIALS = [
    "", "b","p","m","f","d","t","n","l","g","k","h",
    "j","q","x","zh","ch","sh","r","z","c","s"
]

FINALS = [
    "a","ai","an","ang","ao","e","ei","en","eng","er",
    "i","ia","ian","iang","iao","ie","in","ing","iong","iu",
    "o","ong","ou",
    "u","ua","uai","uan","uang","ui","un","uo",
    "v","ve","van","vn"
]

TONES = ["1","2","3","4","5"]

ZERO_INITIAL_MAP = {
    # y-series
    "yi":"i",
    "ya":"ia",
    "yao":"iao",
    "ye":"ie",
    "you":"iu",
    "yan":"ian",
    "yin":"in",
    "yang":"iang",
    "ying":"ing",
    "yong":"iong",

    "yu":"v",
    "yue":"ve",
    "yuan":"van",
    "yun":"vn",

    # w-series
    "wu":"u",
    "wa":"ua",
    "wo":"uo",
    "wai":"uai",
    "wei":"ui",
    "wan":"uan",
    "wen":"un",
    "wang":"uang",
    "weng":"ong",
}


def normalize_zero_initial(base):
    return ZERO_INITIAL_MAP.get(base, base)


def normalize_jqx(final):
    if final.startswith("u"):
        mapping = {
            "u": "v",
            "ue": "ve",
            "uan": "van",
            "un": "vn",
        }
        return mapping.get(final, final)
    return final


init2idx = {v:i for i,v in enumerate(INITIALS)}
final2idx = {v:i for i,v in enumerate(FINALS)}
tone2idx = {v:i for i,v in enumerate(TONES)}

idx2init = {i:v for v,i in init2idx.items()}
idx2final = {i:v for v,i in final2idx.items()}
idx2tone = {i:v for v,i in tone2idx.items()}


def decompose_pinyin(token: str):
    tone = token[-1]

    if not tone.isdigit():
        tone = "5" # neutral tone, usually has no digit
        base = token
    else:
        base = token[:-1]

    base = normalize_zero_initial(base)

    # longest-match for initials

    try:
        for ini in sorted(INITIALS, key=len, reverse=True):
            if base.startswith(ini):
                final = base[len(ini):]

                if ini in {"j","q","x"}:
                    final = normalize_jqx(final)
                
                return [init2idx[ini], final2idx[final], tone2idx[tone]]
    except Exception:
        raise ValueError("Invalid pinyin token")

## Dataset: Text + Pinyin

In [6]:
import pandas as pd

dataset = pd.read_csv("")

In [7]:
dataset.head()

,text,pinyin
0,一个特殊群体的期待在密切关注此次特金会的人中有一个特殊的群体在日本殖民期间来到日本的韩国朝鲜...,yi1 ge4 te4 shu1 qun2 ti3 de qi1 dai4 zai4 mi4...
1,他们希望政治解冻有助于让朝鲜摆脱孤立,ta1 men xi1 wang4 zheng4 zhi4 jie3 dong4 you3 ...
2,他认为对这个奖的妄想可能会促使双方做出轻率的承诺但也可能有助于艰巨的和平进程,ta1 ren4 wei2 dui4 zhe4 ge4 jiang3 de wang4 xi...
3,欢迎阅读纪思道文章的中文版,huan1 ying2 yue4 du2 ji4 si1 dao4 wen2 zhang1 ...
4,不法业者向一些中国女性编造了一个美好的未来到美国开始新生活并拥有一份合法的工作,bu4 fa3 ye4 zhe3 xiang4 yi1 xie1 zhong1 guo2 n...


## Dataset: Pinyin Embeddings

In [8]:
import pickle as pkl

with open("", "rb") as f:
    embedding_map = pkl.load(f)

In [9]:
embedding_map['wo3'].shape

(1024,)

## Training Dataset

First, filter the sequences where at least one token is missing a sound (only `bo` token is expected, with around 70 sentences):

In [10]:
def align(token):
    if token[-1].isalpha():  # e.g. de -> de5
        return token + '5'
    return token

# To see filtered values
dataset[
    dataset['pinyin'].str.split().transform(lambda x: any(align(y) not in embedding_map.keys() for y in x))
]

,text,pinyin
1267,纽约时报现年八十二岁的阿尔及利亚強人总统阿卜杜勒阿齐兹布特弗利卡月底将下台,niu3 yue1 shi2 bao4 xian4 nian2 ba1 shi2 er4 s...
1369,如今安倍的政治对手已经磨刀霍霍安倍经济学命运未卜,ru2 jin1 an1 bei4 de zheng4 zhi4 dui4 shou3 yi...
1440,纽约时报马尔代夫总统易卜拉欣穆罕默德萨利赫所在的政党似乎将在该国议会选举中获得压倒性胜利,niu3 yue1 shi2 bao4 ma3 er3 dai4 fu1 zong3 ton...
2203,两个家庭的命运以悲剧性的方式交汇了波尔森家失去了三个年幼的孩子炸死他们的是易卜拉欣家的两个儿子,liang3 ge4 jia1 ting2 de ming4 yun4 yi3 bei1 j...
4805,阿卜杜拉赫卜表示自己和前夫曾因此收到死亡威胁她希望公开身份能够让自己和家人免于被报复,a1 bo du4 la1 he4 bo biao3 shi4 zi4 ji3 he2 qi...
...,...,...
152505,营地在一条路边那是通往夏河县及著名的庞大藏传佛教寺院拉卜楞寺的道路,ying2 di4 zai4 yi1 tiao2 lu4 bian1 na4 shi4 to...
152511,这两位僧人曾在拉卜楞寺住过寺里的僧人赶来这里迎接他们,zhe4 liang3 wei4 seng1 ren2 ceng2 zai4 la1 bo ...
157326,如果我们收不到货东西就会更贵甚至可能要关门卖黄瓜萝卜和西红柿的左启超音说,ru2 guo3 wo3 men shou1 bu4 dao4 huo4 dong1 xi1...
157327,在他说话的同时一个女人指责他胡来提高萝卜的价格,zai4 ta1 shuo1 hua4 de tong2 shi2 yi1 ge4 nv3 ...


In [11]:
# Actual filtration
dataset = dataset[
    dataset['pinyin'].str.split().transform(lambda x: all(align(y) in embedding_map.keys() for y in x))
]

In [12]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 161392 entries, 0 to 161463
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    161392 non-null  object
 1   pinyin  161392 non-null  object
dtypes: object(2)
memory usage: 3.7+ MB


Next, pre-define the index of pinyin tokens for embedding lookup:

In [13]:
import numpy as np
import torch

all_pinyin_tokens = list(embedding_map.keys())

pinyin2idx = {tok: i for i, tok in enumerate(all_pinyin_tokens)}
embedding_matrix = torch.tensor(np.stack([embedding_map[tok] for tok in all_pinyin_tokens]), dtype=torch.float32)

In [14]:
from tqdm import tqdm 

def tokenize_dataset(dataset, tokenizer, pinyin_to_idx):

    for i, sample in tqdm(dataset.iterrows(), total=len(dataset)):
        text = sample['text']
        pinyin_str = sample['pinyin']

        char_seq = list(text)

        pinyin_tokens = pinyin_str.split()
        
        # Valid sample - yield the indices
        token_idxs, char_span = tokenizer.encode(text)

        pinyin_component_idxs = []
        pinyin_idxs = []
        for token in pinyin_tokens:

            try:
                pinyin_component_idx = decompose_pinyin(token)
                pinyin_component_idxs.append(pinyin_component_idx)

                pinyin_idxs.append(pinyin2idx[align(token)])
            except ValueError:
                print(f"Skipped token {token}")

        
        yield char_seq, token_idxs, char_span, pinyin_component_idxs, pinyin_idxs

preprocessed_dataset = tokenize_dataset(dataset, tokenizer, decompose_pinyin)

In [15]:
char_seqs = []
token_seqs = []
char_spans = []
pinyin_seqs = []
pinyin_idx_seqs = []

for char_seq, token_idxs, char_span, pinyin_component_idxs, pinyin_idxs in preprocessed_dataset:
    char_seqs.append(char_seq)
    token_seqs.append(token_idxs)
    char_spans.append(char_span)
    pinyin_seqs.append(pinyin_component_idxs)
    pinyin_idx_seqs.append(pinyin_idxs)

100%|██████████| 161392/161392 [00:34<00:00, 4722.42it/s]


In [16]:
def build_char_vocab(char_seqs, add_pad=True, add_unk=True):
    """
    Build character vocabulary from a list of character sequences.

    Args:
        char_seqs: list of lists of strings (each inner list is a sentence of characters)
        add_pad: if True, include <PAD> token at index 0
        add_unk: if True, include <UNK> token after pad (if pad exists) or at start

    Returns:
        char2idx: dict mapping character to index
        idx2char: list mapping index to character
    """
    # Collect all unique characters
    chars = set()
    for seq in char_seqs:
        for ch in seq:
            chars.add(ch)
    # Sort for deterministic order
    sorted_chars = sorted(chars)

    # Build vocabulary list
    vocab = []
    if add_pad:
        vocab.append('<PAD>')
    if add_unk:
        vocab.append('<UNK>')
    vocab.extend(sorted_chars)

    # Create mappings
    char2idx = {ch: idx for idx, ch in enumerate(vocab)}
    idx2char = vocab

    return char2idx, idx2char

In [17]:
char2idx, idx2char = build_char_vocab(char_seqs)

In [18]:
char_indices = []

for char_sequence in char_seqs:
    indices = [char2idx[char] for char in char_sequence]
    char_indices.append(indices)

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class PinyinDataset(Dataset):
    """
    Dataset for character‑to‑pinyin with sub‑character tokens, including raw characters,
    their indices, and pinyin embedding indices.
    """
    def __init__(self, char_sequences, char_indices, token_sequences, char_spans,
                 pinyin_sequences, pinyin_idx_sequences):
        """
        Args:
            char_sequences: list of lists of characters (strings) per sample
            char_indices: list of lists of integer indices for each character
            token_sequences: list of lists of token indices (sub‑character tokens)
            char_spans: list of lists of (start, end) tuples for each character
            pinyin_sequences: list of lists of decomposed pinyin vectors (initial, final, tone)
            pinyin_idx_sequences: list of lists of embedding indices (int) for each character
        """
        assert len(char_sequences) == len(char_indices) == len(token_sequences) == len(char_spans) == len(pinyin_sequences) == len(pinyin_idx_sequences)
        self.char_seqs = char_sequences
        self.char_idxs = char_indices
        self.token_seqs = token_sequences
        self.char_spans = char_spans
        self.pinyin_seqs = pinyin_sequences
        self.pinyin_idx_seqs = pinyin_idx_sequences

    def __len__(self):
        return len(self.token_seqs)

    def __getitem__(self, idx):
        return {
            'chars': self.char_seqs[idx],
            'char_idxs': torch.tensor(self.char_idxs[idx], dtype=torch.long),
            'tokens': torch.tensor(self.token_seqs[idx], dtype=torch.long),
            'offsets': torch.tensor([span[0] for span in self.char_spans[idx]], dtype=torch.long),
            'pinyin': torch.tensor(self.pinyin_seqs[idx], dtype=torch.long),      # (num_chars, 3)
            'pinyin_idx': torch.tensor(self.pinyin_idx_seqs[idx], dtype=torch.long) # (num_chars,)
        }

In [20]:
def collate_fn(batch, pad_idx=-100):
    """
    Collate a batch: flatten tokens, build offsets, and collect character lists, indices,
    pinyin component targets, and pinyin embedding indices.
    Returns:
        flat_tokens: (total_tokens,) concatenated token IDs
        char_offsets: (total_chars,) start index of each character in flat_tokens
        char_lengths: (batch,) number of characters per sample
        padded_pinyin: (batch, max_chars, 3) padded pinyin component targets
        chars_list: list of lists of characters (one per sample)
        padded_char_idxs: (batch, max_chars) padded character indices
        padded_pinyin_idxs: (batch, max_chars) padded pinyin embedding indices
    """
    chars_list = [item['chars'] for item in batch]
    char_idxs_list = [item['char_idxs'] for item in batch]
    tokens_list = [item['tokens'] for item in batch]
    offsets_list = [item['offsets'] for item in batch]
    pinyin_list = [item['pinyin'] for item in batch]
    pinyin_idx_list = [item['pinyin_idx'] for item in batch]

    flat_tokens = []
    char_offsets = []
    char_lengths = []
    total_tokens = 0

    for sample_tokens, sample_offsets in zip(tokens_list, offsets_list):
        flat_tokens.extend(sample_tokens.tolist())
        num_chars = len(sample_offsets)
        char_lengths.append(num_chars)
        for i in range(num_chars):
            char_offsets.append(total_tokens + sample_offsets[i].item())
        total_tokens += len(sample_tokens)

    flat_tokens = torch.tensor(flat_tokens, dtype=torch.long)
    char_offsets = torch.tensor(char_offsets, dtype=torch.long)
    char_lengths = torch.tensor(char_lengths, dtype=torch.long)

    padded_pinyin = nn.utils.rnn.pad_sequence(pinyin_list, batch_first=True,
                                              padding_value=pad_idx)
    padded_char_idxs = nn.utils.rnn.pad_sequence(char_idxs_list, batch_first=True,
                                                 padding_value=pad_idx)
    padded_pinyin_idxs = nn.utils.rnn.pad_sequence(pinyin_idx_list, batch_first=True,
                                                   padding_value=pad_idx)

    return (flat_tokens, char_offsets, char_lengths, padded_pinyin, chars_list,
            padded_char_idxs, padded_pinyin_idxs)

In [21]:
def create_dataloaders(char_sequences, char_indices, token_sequences, char_spans,
                       pinyin_sequences, pinyin_idx_sequences, batch_size,
                       val_split=0.1, test_split=0.1, pad_idx=-100, shuffle=True):
    """
    Create train/validation/test DataLoaders.
    Args:
        char_sequences: list of lists of characters (strings) per sample
        char_indices: list of lists of integer indices for each character
        token_sequences: list of token index sequences (per sample)
        char_spans: list of character span lists (per sample)
        pinyin_sequences: list of decomposed pinyin sequences (per sample)
        pinyin_idx_sequences: list of pinyin embedding index sequences (per sample)
        batch_size: batch size
        val_split: fraction of data for validation
        test_split: fraction of data for test
        pad_idx: padding value (e.g., -100)
        shuffle: whether to shuffle training data
    Returns:
        train_loader, val_loader, test_loader
    """
    dataset = PinyinDataset(
        char_sequences, char_indices, token_sequences, char_spans,
        pinyin_sequences, pinyin_idx_sequences
    )

    total = len(dataset)
    val_size = int(total * val_split)
    test_size = int(total * test_split)
    train_size = total - val_size - test_size

    assert train_size > 0, "Not enough data for training split"
    assert val_size >= 0 and test_size >= 0, "Split sizes must be non‑negative"

    lengths = [train_size, val_size, test_size]
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, lengths,
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx)
    )

    return train_loader, val_loader, test_loader

In [22]:
PAD_IDX = -100

train_loader, val_loader, test_loader = create_dataloaders(
    char_seqs, char_indices, token_seqs,
    char_spans, pinyin_seqs, pinyin_idx_seqs,
    batch_size=32, val_split=0.1, test_split=0.1, pad_idx=PAD_IDX
)

# Model Preparation

## Encoder Architecture

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class ClassifierHead(nn.Module):
    """MLP with one hidden layer for classification."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class Encoder(nn.Module):
    """
    BiLSTM model for predicting pinyin from characters.
    - Token embeddings aggregated per character (EmbeddingBag).
    - Character representations passed through BiLSTM.
    - Three MLP heads predict initial, final, and tone.
    """
    def __init__(self, vocab_size_tokens, embedding_dim, hidden_dim,
                 num_initials, num_finals, num_tones,
                 num_layers=2, dropout=0.3, mlp_hidden_dim=128):
        super().__init__()
        self.embedding_bag = nn.EmbeddingBag(vocab_size_tokens, embedding_dim,
                                              mode='mean')
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            bidirectional=True, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

        # MLP heads (input = hidden_dim*2 from BiLSTM)
        self.initial_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_initials, dropout)
        self.final_head   = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_finals, dropout)
        self.tone_head    = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_tones, dropout)

    def forward(self, token_ids, token_offsets, char_lengths, return_hidden=False):
        """
        Args:
            token_ids: (total_tokens,) flattened token IDs (no padding)
            token_offsets: (total_chars,) start indices for each character
            char_lengths: (batch,) number of characters per sample
            return_hidden: if True, also return character‑level hidden states
        Returns:
            logits_initial, logits_final, logits_tone:
                each shape (batch, max_char_len, respective_vocab_size)
            (optional) char_out: (batch, max_char_len, hidden_dim*2) before heads
        """
        char_emb = self.embedding_bag(token_ids, token_offsets)  # (total_chars, emb_dim)

        # Split into per‑sample sequences and pad
        char_emb_list = torch.split(char_emb, char_lengths.tolist())
        padded_char_emb = nn.utils.rnn.pad_sequence(char_emb_list, batch_first=True)

        # Pack for LSTM
        packed_emb = pack_padded_sequence(padded_char_emb, char_lengths.cpu(),
                                          batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed_emb)
        char_out, _ = pad_packed_sequence(packed_out, batch_first=True)
        char_out = self.dropout(char_out)

        # Three MLP heads
        logits_initial = self.initial_head(char_out)
        logits_final   = self.final_head(char_out)
        logits_tone    = self.tone_head(char_out)

        if return_hidden:
            return logits_initial, logits_final, logits_tone, char_out
        return logits_initial, logits_final, logits_tone

In [24]:
vocab_size_tokens = len(tokenizer.vocab)
embedding_dim = 100
hidden_dim = 512
mlp_hidden_dim = 256
num_initials = len(init2idx)
num_finals   = len(final2idx)
num_tones    = len(tone2idx)
num_layers = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create model instance
encoder = Encoder(
    vocab_size_tokens=vocab_size_tokens,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    mlp_hidden_dim=mlp_hidden_dim,
    num_initials=num_initials,
    num_finals=num_finals,
    num_tones=num_tones,
    num_layers=num_layers
).to(device)

## Training Loop

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# Hyperparameters
num_epochs = 20
patience = 5
lambda_mse = 1.0
pad_idx = -100                  # must match collate_fn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Embedding matrix (pre‑computed, on device)
embedding_matrix = embedding_matrix.to(device)   # shape (num_tokens, emb_dim)

# Loss functions
criterion_initial = nn.CrossEntropyLoss(ignore_index=pad_idx)
criterion_final   = nn.CrossEntropyLoss(ignore_index=pad_idx)
criterion_tone    = nn.CrossEntropyLoss(ignore_index=pad_idx)
criterion_mse     = nn.MSELoss()

# Optimizer and scheduler
optimizer = torch.optim.AdamW(encoder.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [26]:
best_val_loss = float('inf')
counter = 0

# For storing metrics
train_losses = []
val_losses = []
val_acc_init = []
val_acc_final = []
val_acc_tone = []
val_acc_exact = []
val_mse = []

for epoch in range(num_epochs):
    # --- Training ---
    encoder.train()
    total_train_loss = 0
    train_steps = 0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for batch in train_pbar:
        # Unpack batch (7 items)
        flat_tokens, char_offsets, char_lengths, padded_pinyin, _, _, padded_pinyin_idxs = batch
        flat_tokens = flat_tokens.to(device)
        char_offsets = char_offsets.to(device)
        char_lengths = char_lengths.to(device)
        padded_pinyin = padded_pinyin.to(device)          # (batch, max_chars, 3)
        padded_pinyin_idxs = padded_pinyin_idxs.to(device)  # (batch, max_chars)

        # Forward with hidden states
        logits_i, logits_f, logits_t, hidden = encoder(
            flat_tokens, char_offsets, char_lengths, return_hidden=True
        )
        # hidden shape: (batch, max_chars, hidden_dim)

        # Cross‑entropy losses
        loss_i = criterion_initial(logits_i.reshape(-1, num_initials), padded_pinyin[..., 0].reshape(-1))
        loss_f = criterion_final(logits_f.reshape(-1, num_finals), padded_pinyin[..., 1].reshape(-1))
        loss_t = criterion_tone(logits_t.reshape(-1, num_tones), padded_pinyin[..., 2].reshape(-1))
        # ce_loss = loss_i + loss_f + loss_t -- IGNORED

        # MSE loss for pronunciation grounding
        # Create mask for valid pinyin indices (non‑padding)
        mask = (padded_pinyin_idxs != pad_idx)  # (batch, max_chars)
        if mask.any():
            valid_hidden = hidden[mask]                       # (num_valid, hidden_dim)
            valid_target_idxs = padded_pinyin_idxs[mask]      # (num_valid,)
            target_embeds = embedding_matrix[valid_target_idxs]  # (num_valid, emb_dim)
            mse_loss = criterion_mse(valid_hidden, target_embeds)
        else:
            mse_loss = torch.tensor(0.0, device=device)

        total_loss = mse_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        total_train_loss += total_loss.item()
        train_steps += 1
        train_pbar.set_postfix({'loss': f'{total_loss.item():.4f}', 'mse': f'{mse_loss.item():.4f}'})

    avg_train_loss = total_train_loss / train_steps
    train_losses.append(avg_train_loss)

    # --- Validation ---
    encoder.eval()
    total_val_loss = 0
    total_val_ce = 0
    total_val_mse = 0
    val_steps = 0
    correct_i = correct_f = correct_t = correct_exact = 0
    total_chars = 0

    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
    with torch.no_grad():
        for batch in val_pbar:
            flat_tokens, char_offsets, char_lengths, padded_pinyin, _, _, padded_pinyin_idxs = batch
            flat_tokens = flat_tokens.to(device)
            char_offsets = char_offsets.to(device)
            char_lengths = char_lengths.to(device)
            padded_pinyin = padded_pinyin.to(device)
            padded_pinyin_idxs = padded_pinyin_idxs.to(device)

            logits_i, logits_f, logits_t, hidden = encoder(
                flat_tokens, char_offsets, char_lengths, return_hidden=True
            )

            # Cross‑entropy loss
            loss_i = criterion_initial(logits_i.reshape(-1, num_initials), padded_pinyin[..., 0].reshape(-1))
            loss_f = criterion_final(logits_f.reshape(-1, num_finals), padded_pinyin[..., 1].reshape(-1))
            loss_t = criterion_tone(logits_t.reshape(-1, num_tones), padded_pinyin[..., 2].reshape(-1))
            ce_loss = loss_i + loss_f + loss_t

            # MSE loss
            mask = (padded_pinyin_idxs != pad_idx)
            if mask.any():
                valid_hidden = hidden[mask]
                valid_target_idxs = padded_pinyin_idxs[mask]
                target_embeds = embedding_matrix[valid_target_idxs]

                hidden_norm = F.normalize(valid_hidden, p=2, dim=-1)
                target_norm = F.normalize(target_embeds, p=2, dim=-1)
                mse_loss = F.mse_loss(hidden_norm, target_norm)
            else:
                mse_loss = torch.tensor(0.0, device=device)

            total_val_ce += ce_loss.item()
            total_val_mse += mse_loss.item()
            total_val_loss += mse_loss.item()
            val_steps += 1

            # Accuracy on pinyin components
            pred_i = logits_i.argmax(dim=-1)
            pred_f = logits_f.argmax(dim=-1)
            pred_t = logits_t.argmax(dim=-1)

            mask_ce = (padded_pinyin[..., 0] != pad_idx)  # same as pinyin indices padding
            correct_i += ((pred_i == padded_pinyin[..., 0]) & mask_ce).sum().item()
            correct_f += ((pred_f == padded_pinyin[..., 1]) & mask_ce).sum().item()
            correct_t += ((pred_t == padded_pinyin[..., 2]) & mask_ce).sum().item()
            exact = (pred_i == padded_pinyin[..., 0]) & (pred_f == padded_pinyin[..., 1]) & (pred_t == padded_pinyin[..., 2]) & mask_ce
            correct_exact += exact.sum().item()
            total_chars += mask_ce.sum().item()

    avg_val_loss = total_val_loss / val_steps
    avg_val_ce = total_val_ce / val_steps
    avg_val_mse = total_val_mse / val_steps

    acc_i = correct_i / total_chars if total_chars else 0
    acc_f = correct_f / total_chars if total_chars else 0
    acc_t = correct_t / total_chars if total_chars else 0
    acc_exact = correct_exact / total_chars if total_chars else 0

    val_losses.append(avg_val_loss)
    val_acc_init.append(acc_i)
    val_acc_final.append(acc_f)
    val_acc_tone.append(acc_t)
    val_acc_exact.append(acc_exact)
    val_mse.append(avg_val_mse)

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val MSE = {avg_val_mse:.4f}")
    print(f"  Init Acc: {acc_i:.4f}, Final Acc: {acc_f:.4f}, Tone Acc: {acc_t:.4f}, Exact: {acc_exact:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(encoder.state_dict(), 'encoder_best.pt')
        print(f"  -> Best encoder saved (val loss: {best_val_loss:.4f})")
        counter = 0
    else:
        counter += 1
        print(f"  -> No improvement for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    scheduler.step(avg_val_loss)

print("Encoder training complete.")

Epoch 1/20 [Val]: 100%|██████████| 505/505 [00:14<00:00, 34.94it/s]


Epoch 1: Train Loss = 0.1947, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.1001, Final Acc: 0.0555, Tone Acc: 0.2072, Exact: 0.0007
  -> Best encoder saved (val loss: 0.0000)


Epoch 2/20 [Val]: 100%|██████████| 505/505 [00:14<00:00, 35.26it/s]


Epoch 2: Train Loss = 0.1753, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.1004, Final Acc: 0.0560, Tone Acc: 0.2071, Exact: 0.0006
  -> Best encoder saved (val loss: 0.0000)


Epoch 3/20 [Val]: 100%|██████████| 505/505 [00:14<00:00, 35.31it/s]


...

Epoch 18: Train Loss = 0.1682, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.0954, Final Acc: 0.0598, Tone Acc: 0.2072, Exact: 0.0010
  -> Best encoder saved (val loss: 0.0000)


Epoch 19/20 [Val]: 100%|██████████| 505/505 [00:14<00:00, 35.32it/s]


Epoch 19: Train Loss = 0.1681, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.0952, Final Acc: 0.0599, Tone Acc: 0.2072, Exact: 0.0009
  -> No improvement for 1 epoch(s).


Epoch 20/20 [Val]: 100%|██████████| 505/505 [00:14<00:00, 35.41it/s]


Epoch 20: Train Loss = 0.1681, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.0978, Final Acc: 0.0600, Tone Acc: 0.2072, Exact: 0.0010
  -> Best encoder saved (val loss: 0.0000)
Encoder training complete.


In [27]:
import pickle
with open('training_metrics.pkl', 'wb') as f:
    pickle.dump({
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_acc_init': val_acc_init,
        'val_acc_final': val_acc_final,
        'val_acc_tone': val_acc_tone,
        'val_acc_exact': val_acc_exact,
        'val_mse': val_mse
    }, f)

In [28]:
def evaluate_encoder(encoder, dataloader, device):
    """
    Evaluate encoder on a dataloader (test/validation).
    Assumes dataloader returns (flat_tokens, char_offsets, char_lengths, padded_pinyin, chars_list, padded_char_idxs)
    where padded_pinyin has shape (batch, max_chars, 3) and padding value = -100.
    Returns a dict of accuracies: initial, final, tone, exact.
    """
    encoder.eval()
    total_initial_correct = 0
    total_final_correct = 0
    total_tone_correct = 0
    total_exact_correct = 0
    total_chars = 0

    with torch.no_grad():
        for batch in tqdm(dataloader):
            # Unpack the first 4 elements; ignore the rest
            flat_tokens, char_offsets, char_lengths, padded_pinyin = batch[:4]
            flat_tokens = flat_tokens.to(device)
            char_offsets = char_offsets.to(device)
            char_lengths = char_lengths.to(device)
            padded_pinyin = padded_pinyin.to(device)  # (batch, max_chars, 3)

            logits_i, logits_f, logits_t = encoder(flat_tokens, char_offsets, char_lengths)
            # logits: (batch, max_chars, vocab_size)

            pred_i = logits_i.argmax(dim=-1)
            pred_f = logits_f.argmax(dim=-1)
            pred_t = logits_t.argmax(dim=-1)

            # Mask for valid characters (where pinyin is not padding)
            mask = padded_pinyin[:, :, 0] != -100  # (batch, max_chars)

            # Component‑wise correctness
            correct_i = (pred_i == padded_pinyin[:, :, 0]) & mask
            correct_f = (pred_f == padded_pinyin[:, :, 1]) & mask
            correct_t = (pred_t == padded_pinyin[:, :, 2]) & mask

            total_initial_correct += correct_i.sum().item()
            total_final_correct   += correct_f.sum().item()
            total_tone_correct    += correct_t.sum().item()

            exact_match = correct_i & correct_f & correct_t
            total_exact_correct += exact_match.sum().item()

            total_chars += mask.sum().item()

    acc_initial = total_initial_correct / total_chars if total_chars > 0 else 0
    acc_final   = total_final_correct   / total_chars
    acc_tone    = total_tone_correct    / total_chars
    acc_exact   = total_exact_correct   / total_chars

    return {
        'initial_accuracy': acc_initial,
        'final_accuracy':   acc_final,
        'tone_accuracy':    acc_tone,
        'exact_accuracy':   acc_exact
    }

In [29]:
# Load the best encoder from stage 1
encoder.load_state_dict(torch.load('encoder_best.pt', map_location=device))
encoder.eval()

Encoder(
  (embedding_bag): EmbeddingBag(39, 100, mode='mean')
  (lstm): LSTM(100, 512, num_layers=4, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (initial_head): ClassifierHead(
    (fc1): Linear(in_features=1024, out_features=256, bias=True)
    (act): ReLU()
    (dropout): Dropout(p=0.3, inplace=False)
    (fc2): Linear(in_features=256, out_features=22, bias=True)
  )
  (final_head): ClassifierHead(
    (fc1): Linear(in_features=1024, out_features=256, bias=True)
    (act): ReLU()
    (dropout): Dropout(p=0.3, inplace=False)
    (fc2): Linear(in_features=256, out_features=35, bias=True)
  )
  (tone_head): ClassifierHead(
    (fc1): Linear(in_features=1024, out_features=256, bias=True)
    (act): ReLU()
    (dropout): Dropout(p=0.3, inplace=False)
    (fc2): Linear(in_features=256, out_features=5, bias=True)
  )
)

In [30]:
metrics = evaluate_encoder(encoder, test_loader, device)
print("Test metrics:", metrics)

100%|██████████| 505/505 [00:13<00:00, 36.71it/s]

Test metrics: {'initial_accuracy': 0.09786975510934881, 'final_accuracy': 0.05971083067279044, 'tone_accuracy': 0.2070013016039704, 'exact_accuracy': 0.0009314414811327777}
